In [ ]:
from google.colab import drive
drive.mount("<mount-point>")

In [ ]:
from sklearn.model_selection import KFold

import pandas as pd
import numpy as np
import datasets

import warnings
warnings.simplefilter(action='ignore')

head_path = '<project-data-path>'
default_type = 1
full_rank = pd.read_excel(f'{head_path}194_full_C_ord.xlsx', index_col=0)

## Loading data

In [ ]:
# true_index = full_rank['true_index'].apply(
#         lambda x:
#         x.replace('[','').replace(']','').split()
#     ) \
#     .apply(
#         lambda x: [int(i) for i in x]
#     ) \
#     .values

# true_index = np.concat(true_index).reshape(-1,3)

# def get_ordered_rank(rank_df):
#     distractors = rank_df.loc[:,['distractor_1', 'distractor_2', 'distractor_3']].values
#     ranks = rank_df.loc[:,['d1_rank', 'd2_rank', 'd3_rank']].values

#     ordered_distractors = distractors.copy()
#     ordered_ranks = ranks.copy()

#     for idx in range(distractors.shape[0]):
#         ordered_distractors[idx] = distractors[idx, np.argsort(true_index[idx,:])]
#         ordered_ranks[idx] = ranks[idx, np.argsort(true_index[idx,:])]

#     invalid_ranks = np.argwhere(ordered_ranks == 4)

#     print(ordered_ranks.shape)
#     print(invalid_ranks.shape)
#     print(np.unique(invalid_ranks[:,1],return_counts = True))

#     return ordered_distractors, ordered_ranks

# ord_distractors, _ = get_ordered_rank(full_rank)

In [ ]:
ord_distractors = full_rank.loc[:,['distractor_1', 'distractor_2', 'distractor_3']].values
ord_distractors

In [ ]:
import os
from openai import OpenAI, AzureOpenAI
import re

# CityU client
cityu_client = AzureOpenAI(
  azure_endpoint=os.environ.get("AZURE_OPENAI_ENDPOINT", "https://<your-azure-openai-endpoint>"),
  api_key=os.environ["AZURE_OPENAI_API_KEY"],
  api_version="2024-02-15"
)

# Personal client
chat_client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ.get("OPENAI_BASE_URL", "https://api.chatanywhere.tech/v1")
)

try:
  response = cityu_client.chat.completions.create(
  model="gpt-4o-ca",
  messages=[
      {"role": "user", "content": 'hello'}
      ],
  max_tokens=1,
  temperature=0.6,
  top_p=0.9
  )
  client = cityu_client
  print('Using CityU API')
except:
  response = chat_client.chat.completions.create(
  model="gpt-4o-ca",
  messages=[
      {"role": "user", "content": 'hello'}
      ],
  max_tokens=1,
  temperature=0.6,
  top_p=0.9
  )
  client = chat_client
  print('Using ChatAnywhere API')

In [ ]:
output = response.choices[0].message.content.lstrip().rstrip().replace('\n', ' ')
output

## Preparing inference

In [ ]:
def pseudo_output(options):
    return str(np.random.choice(options,1)[0])

pseudo_output(['A', 'B', 'C', 'D'])

In [ ]:
def get_shuffled_options(options):
    option_idx = ['berzak','zero','sft']
    option_dict = dict()
    option_map_dict = dict()

    shuffled_idx = np.random.permutation([0, 1, 2])
    # np.random.shuffle(option_idx)
    idx = 'A'
    for s_idx in shuffled_idx:
        option_dict[idx] = options[s_idx]
        option_map_dict[idx] = option_idx[s_idx]
        idx = chr(ord(idx)+1)

    compete_prompt = ''
    for k in ['A', 'B', 'C']:
        v = option_dict.get(k)
        if v is not None:
            compete_prompt += f'{k}. {v}\n'

    return option_dict, option_map_dict, compete_prompt

option_dict, option_map_dict, compete_prompt = get_shuffled_options(ord_distractors[0])

response = pseudo_output(['A', 'B', 'C'])
# response = 'None'

if response in option_dict.keys():
    response_label = option_map_dict[response]
else:
    response_label = response

print(compete_prompt)
print(option_dict)
print(option_map_dict)
print(response, response_label)

## Plain output

In [ ]:
def get_prompts(passage, question, compete_prompt):
    prompt_format = f"""
    The following is a reading comprehension passage:
    Passage: {passage}

    The following is a question from a multiple-choice reading comprehension task:
    Question: {question}

    The following are three potential answers to the question based on the information given in the passage:
    {compete_prompt}
    """

    best_only_prompt = prompt_format + """Based on the given passage as evidence, which option feels like the correct answer to the question? Respond only with A, B, C. Do not give any explanation or rationale."""

    rank_prompt = prompt_format + """Although the distractors are incorrect, please rank these distractors according to the following criteria:

    1. Textual overlap (Most important)\nA good distractor should contain more textual overlaps with the given passage, making it look like it came directly from the text, even if it misrepresents the information.
    2. Semantic similarity (More important)\nA good distractor should match the meaning or ideas of the passage, aligning with its content even if it's incorrect.
    3. Logical inference (Less important)\nA good distractor should seem like a logical conclusion from the passage, even if it involves slight contradictions or errors.

    Again, your judgement should objective and only based on the given passage, instead of your intuitive logic.

    First, evaluate the options one by one according to the above criteria. Give your explanation with no more than 100 words.\n\nThen, according to your rationale, rank the distractors by plausibility and output the rank using only the letter indices separated by comma and nothing else."""

    best_only_prompt = best_only_prompt.replace('\n    ', '\n').strip()
    rank_prompt = rank_prompt.replace('\n    ', '\n').strip()

    return best_only_prompt, rank_prompt

In [ ]:
def get_improved_prompts(passage, question, compete_prompt, type = default_type):
    """
    type: [0,1], 0 for B and 1 for C
    """
    prompt_format = f"""
    The following is a reading comprehension passage:
    Passage: {passage}

    The following is a question from a multiple-choice reading comprehension task:
    Question: {question}

    The following are three potential answers to the question based on the information given in the passage:
    {compete_prompt}
    """

    definition = [
        "incorrect answers which represent plausible misunderstanding of the correct span.",
        "incorrect answers which represent plausible misunderstanding of the wrong information."
    ]

    preference = [
    """1. Semantic similarity (Most important)\nA good distractor should match the meaning of the passage, especially the correct span, even if it slightly misrepresents details due to misunderstandings.
    2. Textual overlap (More important)\nA good distractor might share some wording with the passage, but overlap is less critical since misunderstandings can still seem correct.
    3. Logical inference (Less important)\nA good distractor can have errors or contradictions occasionally, as misunderstandings don't require strong logical connections to the passage.""",
    """1. Textual overlap (Most important)\nA good distractor can contain more textual overlaps with the given passage, making it look like it came directly from the text.
    2. Semantic similarity (More important)\nA good distractor can match the meaning or ideas of the passage, aligning with its content even if it's incorrect.
    3. Logical inference (Less important)\nA good distractor can seem like a logical conclusion from the passage, even if it involves slight contradictions or errors."""
    ]

    ending = [
        'Based on the given passage as evidence, which option feels like the correct answer to the question? Respond only with A, B, C. Do not give any explanation or rationale.',
        'Again, your judgement should objective and only based on the given passage, instead of your intuitive logic.\n\nFirst, evaluate the options one by one according to the above criteria. Give your explanation with no more than 100 words.\n\nThen, according to your rationale, rank the distractors by their plausibility and output the rank using only the letter indices separated by comma and nothing else.'
    ]

    best_only_prompt = prompt_format + f"""All these distractors are {definition[type]}, please rank these distractors according to the following criteria:
    {preference[type]}\n\n{ending[0]}"""

    rank_prompt = prompt_format + f"""All these distractors are {definition[type]}, please rank these distractors according to the following criteria:
    {preference[type]}\n\n{ending[1]}"""

    best_only_prompt = best_only_prompt.replace('\n    ', '\n').strip()
    rank_prompt = rank_prompt.replace('\n    ', '\n').strip()

    return best_only_prompt, rank_prompt

In [ ]:
# Replace the function here
get_prompts = get_improved_prompts

print(get_prompts('passage here', 'question here', compete_prompt)[1])

In [ ]:
articles = full_rank[['article']].values
questions = full_rank[['question']].values
ord_distractors = ord_distractors

test_idx = 0
option_dict, option_map_dict, compete_prompt = get_shuffled_options(ord_distractors[test_idx])
best_only_prompt, rank_prompt = get_prompts(articles[test_idx][0], questions[test_idx][0], compete_prompt)

# best_only_response = client.chat.completions.create(
#   model="gpt-4o-ca",
#   messages=[
#       {"role": "user", "content": best_only_prompt}
#       ],
#   max_tokens=1,
#   temperature=0.6,
#   top_p=0.9
#   )

# rank_response = client.chat.completions.create(
#   model="gpt-4o-ca",
#   messages=[
#       {"role": "user", "content": rank_prompt}
#       ],
#   max_tokens=200,
#   temperature=0.6,
#   top_p=0.9
#   )

# best_only_choice = best_only_response.choices[0].message.content
# best_only_label = option_map_dict[best_only_choice]
# rank_response = rank_response.choices[0].message.content

# print(option_map_dict)
# print(best_only_choice, best_only_label)

In [ ]:
# import json
# from tqdm import tqdm
# from pathlib import Path
# from collections import defaultdict

# output_path = Path(f'{full_rank.shape[0]}_ranking_B_results.json')

# # Initialize or load existed data
# if output_path.exists():
#     with open(output_path, "r", encoding='utf-8') as f:
#         existing_data = json.load(f)
#     current_results = defaultdict(dict, {int(k): v for k, v in existing_data.items()})
#     processed_indices = list(current_results.keys())
# else:
#     current_results = defaultdict(dict)
#     processed_indices = list()

# try:
#     for outer_idx in tqdm(range(0, full_rank.shape[0]), desc="Processing"):
#         if outer_idx in processed_indices:
#             continue

#         # Get article and question
#         attempt = 0
#         while attempt < 4:
#             option_dict, option_map_dict, compete_prompt = get_shuffled_options(ord_distractors[outer_idx])
#             best_only_prompt, rank_prompt = get_prompts(
#                 articles[outer_idx][0],
#                 questions[outer_idx][0],
#                 compete_prompt
#             )

#             best_only_response = client.chat.completions.create(
#                 model="gpt-4o-ca",
#                 messages=[
#                     {"role": "user", "content": best_only_prompt}
#                     ],
#                 max_tokens=1,
#                 temperature=0.6,
#                 top_p=0.9
#                 )

#             rank_response = client.chat.completions.create(
#                 model="gpt-4o-ca",
#                 messages=[
#                     {"role": "user", "content": rank_prompt}
#                     ],
#                 max_tokens=250,
#                 temperature=0.6,
#                 top_p=0.9
#                 )

#             best_only_choice = best_only_response.choices[0].message.content

#             if best_only_choice in option_map_dict.keys():
#                 best_only_label = option_map_dict[best_only_choice]
#             else:
#                 best_only_label = best_only_choice

#             rank_response = rank_response.choices[0].message.content

#             current_results[outer_idx][attempt] = {
#                 'option_dict': option_dict,
#                 'option_map_dict': option_map_dict,
#                 'best_only_response': best_only_choice,
#                 'best_only_label': best_only_label,
#                 'rank_response': rank_response
#             }

#             attempt += 1

#         processed_indices.append(outer_idx)

#         with open(output_path, "w", encoding='utf-8') as f:
#             json.dump(dict(current_results), f, indent=2, ensure_ascii=False)

# except Exception as e:
#     print(f"Error, but save to {output_path}")
#     raise

In [ ]:
print(rank_prompt)

In [ ]:
import json
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict
import torch
import nest_asyncio  # 关键修复包
import asyncio

# 允许在 Colab 中嵌套运行事件循环
nest_asyncio.apply()

# GPU 设置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

output_path = Path(f'{head_path}{full_rank.shape[0]}_ranking_C_results_exp3.json')

# 初始化或加载现有数据
if output_path.exists():
    with open(output_path, "r", encoding='utf-8') as f:
        existing_data = json.load(f)
    current_results = defaultdict(dict, {int(k): v for k, v in existing_data.items()})
    processed_indices = list(current_results.keys())
else:
    current_results = defaultdict(dict)
    processed_indices = []

# 批处理参数
BATCH_SIZE = 20
NUM_WORKERS = 4

async def process_single_index(outer_idx):
    """处理单个索引（异步版本）"""
    attempts_data = {}
    for attempt in range(4):
        try:
            # GPU 数据传输（如果是张量）
            option_dict, option_map_dict, compete_prompt = get_shuffled_options(
                torch.tensor(ord_distractors[outer_idx]).to(device)
                if torch.is_tensor(ord_distractors[outer_idx])
                else ord_distractors[outer_idx]
            )

            best_only_prompt, rank_prompt = get_prompts(
                torch.tensor(articles[outer_idx][0]).to(device)
                if torch.is_tensor(articles[outer_idx][0])
                else articles[outer_idx][0],
                torch.tensor(questions[outer_idx][0]).to(device)
                if torch.is_tensor(questions[outer_idx][0])
                else questions[outer_idx][0],
                compete_prompt
            )

            # 异步 API 调用
            best_only_response = await asyncio.to_thread(
                client.chat.completions.create,
                model="gpt-4o-ca",
                messages=[{"role": "user", "content": best_only_prompt}],
                max_tokens=1,
                temperature=0.6,
                top_p=0.9
            )

            rank_response = await asyncio.to_thread(
                client.chat.completions.create,
                model="gpt-4o-ca",
                messages=[{"role": "user", "content": rank_prompt}],
                max_tokens=250,
                temperature=0.6,
                top_p=0.9
            )

            best_only_choice = best_only_response.choices[0].message.content
            attempts_data[attempt] = {
                'option_dict': option_dict,
                'option_map_dict': option_map_dict,
                'best_only_response': best_only_choice,
                'best_only_label': option_map_dict.get(best_only_choice, best_only_choice),
                'rank_response': rank_response.choices[0].message.content
            }

        except Exception as e:
            attempts_data[attempt] = {"error": str(e)}

    return outer_idx, attempts_data

async def process_batch(batch_indices):
    """异步处理批次数据"""
    tasks = [process_single_index(outer_idx) for outer_idx in batch_indices]
    return await asyncio.gather(*tasks)

def run_in_colab():
    """Colab专用运行函数"""
    unprocessed_indices = [i for i in range(full_rank.shape[0]) if i not in processed_indices]
    total_batches = (len(unprocessed_indices) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num in tqdm(range(total_batches), desc="Processing batches"):
        batch_start = batch_num * BATCH_SIZE
        batch_end = min(batch_start + BATCH_SIZE, len(unprocessed_indices))
        batch_indices = unprocessed_indices[batch_start:batch_end]

        # 在现有事件循环中运行
        batch_results = asyncio.get_event_loop().run_until_complete(
            process_batch(batch_indices)
        )

        # 更新结果
        for outer_idx, result in batch_results:
            current_results[outer_idx] = result
            processed_indices.append(outer_idx)

        # 保存进度
        with open(output_path, "w", encoding='utf-8') as f:
            json.dump(dict(current_results), f, indent=2, ensure_ascii=False)

# 在 Colab 中执行
try:
    run_in_colab()
except Exception as e:
    print(f"Error occurred: {str(e)}")
    print(f"Saving progress to {output_path}")
    with open(output_path, "w", encoding='utf-8') as f:
        json.dump(dict(current_results), f, indent=2, ensure_ascii=False)
    raise

print("Processing completed successfully!")

## Structured output

In [ ]:
# !pip install structured_logprobs

In [ ]:
# from openai.types import ResponseFormatJSONSchema
# from structured_logprobs.main import add_logprobs, add_logprobs_inline

# from collections import defaultdict

In [ ]:
# schema_content = {
#     "type": "json_schema",
#     "json_schema": {
#         "name": "answears",
#         "description": "Response to questions in JSON format",
#         "schema": {
#             "type": "object",
#             "properties": {
#                 "answer_questions": {
#                     "type": "array",
#                     "items": {
#                         "type": "string",
#                         "enum": ["A", "B", "C"]},
#                 },
#             },
#             "required": ["answer_questions"],
#             "additionalProperties": False,
#             },
#             "strict": True,
#         },
#     }

# response_schema = ResponseFormatJSONSchema.model_validate(schema_content)

In [ ]:
# def get_conf_rank(option_dict, option_map_dict, best_only_prompt):
#     conf_response = client.chat.completions.create(
#                 model="gpt-4o-ca",
#                 messages = [
#                         {
#                             "role": "system",
#                             "content": best_only_prompt,
#                         }
#                     ],
#                 logprobs=True,
#                 seed=42,
#                 n=4,
#                 response_format=response_schema.model_dump(by_alias=True),
#             )

#     conf_content = add_logprobs_inline(conf_response)
#     conf_logprob = add_logprobs(conf_response)

#     label_probs = defaultdict(list)
#     for c in range(4):
#         match_idx = re.search(r'["\']([A-Z])["\']', conf_content.choices[c].message.content)
#         idx = match_idx.group(1) if match_idx else None
#         if idx in option_map_dict:
#             label = option_map_dict[idx]
#             prob = np.exp(conf_logprob.log_probs[c]['answer_questions'][0])
#             label_probs[label].append(prob)

#     # Detecting repeated option
#     value_to_labels = defaultdict(list)
#     for letter, option in option_dict.items():
#         label = option_map_dict[letter]
#         value_to_labels[option].append(label)

#     results = {}

#     # Due with repeated option
#     for option, labels in value_to_labels.items():
#         if len(labels) > 1:
#             all_probs = []
#             for label in labels:
#                 if label in label_probs:
#                     all_probs.extend(label_probs[label])

#             if all_probs:
#                 avg_prob = sum(all_probs) / len(all_probs)
#                 for label in labels:
#                     results[label] = {
#                         'index': [k for k, v in option_map_dict.items() if v == label][0],
#                         'prob': avg_prob
#                     }
#         else:
#             label = labels[0]
#             if label in label_probs:
#                 results[label] = {
#                     'index': [k for k, v in option_map_dict.items() if v == label][0],
#                     'prob': sum(label_probs[label]) / len(label_probs[label])
#                 }

#     required_labels = set(option_map_dict.values())
#     missing_labels = [label for label in required_labels if label not in results]
#     missing_count = len(missing_labels)

#     if missing_count > 0:
#         exist_probs = np.array([v['prob'] for v in results.values()])
#         remaining_prob = 1 - exist_probs.sum()

#         if remaining_prob > 0:
#             avg_prob = remaining_prob / missing_count
#             avg_prob = max(1e-6, avg_prob)

#             for label in missing_labels:
#                 results[label] = {
#                     'index': 'E',
#                     'prob': avg_prob
#                 }
#         else:
#             for label in missing_labels:
#                 results[label] = {
#                     'index': 'E',
#                     'prob': 1e-6
#                 }

#     return results

In [ ]:
# import json
# from tqdm import tqdm
# from pathlib import Path
# from collections import defaultdict
# import torch
# import nest_asyncio  # 关键修复包
# import asyncio

# # 允许在 Colab 中嵌套运行事件循环
# nest_asyncio.apply()

# # GPU 设置
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# output_path = Path(f'<project-data-path>')

# # 初始化或加载现有数据
# if output_path.exists():
#     with open(output_path, "r", encoding='utf-8') as f:
#         existing_data = json.load(f)
#     current_results = defaultdict(dict, {int(k): v for k, v in existing_data.items()})
#     processed_indices = list(current_results.keys())
# else:
#     current_results = defaultdict(dict)
#     processed_indices = []

# # 批处理参数
# BATCH_SIZE = 20
# NUM_WORKERS = 4

# async def process_single_index(outer_idx):
#     """处理单个索引（异步版本）"""
#     attempts_data = {}
#     for attempt in range(4):
#         try:
#             # GPU 数据传输（如果是张量）
#             option_dict, option_map_dict, compete_prompt = get_shuffled_options(
#                 torch.tensor(ord_distractors[outer_idx]).to(device)
#                 if torch.is_tensor(ord_distractors[outer_idx])
#                 else ord_distractors[outer_idx]
#             )

#             best_only_prompt, rank_prompt = get_prompts(
#                 torch.tensor(articles[outer_idx][0]).to(device)
#                 if torch.is_tensor(articles[outer_idx][0])
#                 else articles[outer_idx][0],
#                 torch.tensor(questions[outer_idx][0]).to(device)
#                 if torch.is_tensor(questions[outer_idx][0])
#                 else questions[outer_idx][0],
#                 compete_prompt
#             )

#             # 异步 API 调用
#             conf_rank_results = await asyncio.to_thread(
#                 get_conf_rank,
#                 option_dict = option_dict,
#                 option_map_dict = option_map_dict,
#                 best_only_prompt = best_only_prompt
#             )

#             attempts_data[attempt] = {
#                 'option_dict': option_dict,
#                 'option_map_dict': option_map_dict,
#                 'conf_rank_dict': conf_rank_results,
#             }

#         except Exception as e:
#             attempts_data[attempt] = {"error": str(e)}

#     return outer_idx, attempts_data

# async def process_batch(batch_indices):
#     """异步处理批次数据"""
#     tasks = [process_single_index(outer_idx) for outer_idx in batch_indices]
#     return await asyncio.gather(*tasks)

# def run_in_colab():
#     """Colab专用运行函数"""
#     unprocessed_indices = [i for i in range(full_rank.shape[0]) if i not in processed_indices]
#     total_batches = (len(unprocessed_indices) + BATCH_SIZE - 1) // BATCH_SIZE

#     for batch_num in tqdm(range(total_batches), desc="Processing batches"):
#         batch_start = batch_num * BATCH_SIZE
#         batch_end = min(batch_start + BATCH_SIZE, len(unprocessed_indices))
#         batch_indices = unprocessed_indices[batch_start:batch_end]

#         # 在现有事件循环中运行
#         batch_results = asyncio.get_event_loop().run_until_complete(
#             process_batch(batch_indices)
#         )

#         # 更新结果
#         for outer_idx, result in batch_results:
#             current_results[outer_idx] = result
#             processed_indices.append(outer_idx)

#         # 保存进度
#         with open(output_path, "w", encoding='utf-8') as f:
#             json.dump(dict(current_results), f, indent=2, ensure_ascii=False)

# # 在 Colab 中执行
# try:
#     run_in_colab()
# except Exception as e:
#     print(f"Error occurred: {str(e)}")
#     print(f"Saving progress to {output_path}")
#     with open(output_path, "w", encoding='utf-8') as f:
#         json.dump(dict(current_results), f, indent=2, ensure_ascii=False)
#     raise

# print("Processing completed successfully!")